# Zhai et al. (2025) on dandiset 001538

Paper: Zhai, Cui, Wokosin, Sun, Tkatch, Crittenden, Graybiel and Surmeier (2025), "State-dependent modulation of spiny projection neurons controls levodopa-induced dyskinesia in a mouse model of Parkinson's disease", Science Advances 11, eadv8224, doi:10.1126/sciadv.adv8224. The claims file (`claims/zhai-2025.yaml`) was written from the full text of the version of record in PubMed Central (PMC12674127), its Results, figure legends and Materials and Methods.

Dataset: DANDI:001538, version `draft`. This attempt was asked to use the draft (a repeat attempt on data that changed after an earlier one). A draft can change: the archive gives its last modification as 2026-11-09T19:30:00Z, printed below as the notebook reads it. The draft holds 1288 assets (391 NWB files and the AIM session videos); the latest published version, 0.260527.1302, holds 468 and predates the changes.

Outside inputs: the paper's per-cell and per-mouse tables (Zenodo doi:10.5281/zenodo.17307059), used by the link claims and by the claims the claims file marks `data: elsewhere`. Method details come from the paper's analysis scripts, cited in `lid.py` (doi:10.5281/zenodo.14145776, 14163502 and 14163731).

No step uses random numbers.

| Claim | Data | Tested here |
|---|---|---|
| link-somatic-cells | with inputs | tested |
| link-dendritic-cells | with inputs | tested |
| link-sr-cells | with inputs | tested |
| link-grab-rois | with inputs | tested |
| link-aim-mice | with labels | tested |
| fig1f-dspn-rheobase-on-lower, fig1e-dspn-fi-on-higher, fig1f-dspn-sch-no-change | dataset | tested |
| fig1i-dspn-dendritic-on-higher, fig1i-dspn-dendritic-sch-lower | dataset | tested |
| fig2h-dspn-sr-amplitude-on-larger, fig2i-dspn-sr-frequency-unchanged | dataset | tested |
| fig3d-ispn-rheobase-on-higher, fig3c-ispn-fi-on-lower, fig3d-ispn-sul-no-change | dataset | tested |
| fig3f-ispn-dendritic-on-lower, fig3f-ispn-dendritic-sul-no-change | dataset | tested |
| fig4g-ispn-sr-amplitude-on-smaller, fig4h-ispn-sr-frequency-unchanged | dataset | tested |
| fig5f-ach-off-elevated, fig5f-ach-off-like-6ohda, fig5f-ach-6ohda-elevated, fig5f-ach-d2-control, fig5f-ach-da-suppresses | dataset | tested |
| fig6c-m1r-antag-somatic-unchanged, fig6d-m1r-antag-dendritic-lower | dataset | tested |
| fig7c-cdgi-ko-somatic-oscillation, fig7f-cdgi-ko-dendritic-unchanged | dataset | tested |
| fig9d-m1r-crispr-somatic-unchanged | dataset | tested |
| fig10b-m1r-aims | with labels | tested |
| fig1-dspn-lesion-and-off-somatic, fig1-dspn-lesion-dendritic-unchanged, fig3-ispn-lesion-and-off-somatic, fig3-ispn-lesion-dendritic-up, fig7e-oxom-cdgi, fig8a-cdgi-rotations, fig10a-m1r-rotations, fig10ef-open-field | elsewhere (paper's tables) | tested |
| fig2bc-dspn-spines, fig4bc-ispn-spines, fig6f-m1r-antag-spines, fig7h-cdgi-ko-spines, fig9f-m1r-crispr-spines | dataset | blocked: methods underspecified |
| fig4k-confocal-spines | dataset | blocked: methods underspecified |
| fig8bc-cdgi-aims | with labels | blocked: data missing (the published Figure 8 scores and five of its mice are not in the dandiset) |

The spine-density claims need spine counts. The dandiset holds the images but no counts: 76 two-photon Z-stack files of dendritic segments (Figures 2, 4, 6, 7 and 9) and 16 confocal Z-stack files of sparsely labelled iSPNs with one overview image (Figure 4I to 4K), counted in the data check below. The paper counted spines on the two-photon stacks after deconvolution in AutoQuant X3, semi-automatically in NeuronStudio with manual correction (Methods, "Imaging of spines and dendritic Ca2+ transients"), and on the confocal stacks in Imaris 10 with a trained pixel classifier, supervised filament and spine detection and manual correction (Methods, "High-resolution confocal microscopy"). The manual corrections are not described, so no automatic count has a basis in the paper, and the dandiset description says the rows of the paper's spine tables cannot be matched to the files. These claims are blocked as methods underspecified.

In [ ]:
import time, resource, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import lid

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 400)
T0 = time.time()
info = lid.version_info()
print("dandiset", lid.DANDISET, "version", lid.VERSION, "| modified", info["modified"],
      "| assets", info["asset_count"], "| status", info["status"])
assets = lid.list_assets()
print(len(assets), "NWB files (videos are external mp4 files, not counted)")
print(assets.groupby(["cell_type", "state", "drug", "genotype", "kind"]).size().to_string())

## Loading

Each file is streamed once and reduced as it loads: spike counts per sweep and the line-scan profiles (patch clamp), event amplitudes (Sr2+-oEPSCs), the mean of the 50 um central circle per frame (GRAB-ACh3.0) and the AIM ratings. The paper's per-cell tables (Zenodo doi:10.5281/zenodo.17307059) are read in memory; they are used only by the link claims and the claims the claims file marks `data: elsewhere`.

In [ ]:
t = time.time()
patch = assets[assets.kind.isin(["icephys", "icephys+ophys"]) & assets.path.str.contains("SPN")]
# Sr2+-oEPSC files are the voltage-clamp files whose session description says so; the other icephys files
# hold current-step recordings (printed in the data check).
ic_desc = pd.Series({i: lid.session_description(a.asset_id) for i, a in patch[patch.kind == "icephys"].iterrows()})
sr_files = patch.loc[ic_desc[ic_desc.str.startswith("Sr2+-oEPSC")].index]
patch = patch.drop(sr_files.index)
som_list, den_list = [], []
for _, a in patch.iterrows():
    s, d = lid.load_patch_file(a)
    som_list.append(s); den_list.append(d)
somatic = pd.concat(som_list, ignore_index=True)
dendritic = pd.concat(den_list, ignore_index=True)
print(f"patch-clamp files: {len(patch)}, {time.time() - t:.0f} s")
t = time.time()
sr = pd.concat([lid.load_sr_file(a, (3, 4, 5)) for _, a in sr_files.iterrows()], ignore_index=True)
print(f"Sr2+-oEPSC files: {len(sr_files)}, {time.time() - t:.0f} s")
t = time.time()
grab_files = assets[assets.kind == "ophys"]
grab_trials = pd.concat([lid.load_grab_file(a) for _, a in grab_files.iterrows()], ignore_index=True)
print(f"GRAB-ACh3.0 files: {len(grab_files)}, {time.time() - t:.0f} s")
t = time.time()
aim_files = assets[(assets.cell_type == "pan") & (assets.kind != "ophys")]
aim = pd.concat([lid.load_aim_file(a) for _, a in aim_files.iterrows()], ignore_index=True)
print(f"AIM files: {len(aim_files)}, {time.time() - t:.0f} s")
tables = lid.paper_tables()
print("paper tables:", sorted(tables))
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
fi = lid.cell_fi(somatic)
fi_last = lid.cell_fi(somatic, keep="last")   # variant: a step recorded twice keeps its last sweep
den_cells, den_pairs, den_sites = lid.dendritic_index(dendritic)
grab = lid.grab_auc(grab_trials)
KEYS = ["path", "subject", "date", "cell_type", "file_state", "cell", "condition", "drug"]
# every cell with dendritic line scans, with its index where it has a proximal-distal pair
den_all = den_sites[KEYS].drop_duplicates().merge(den_cells, on=KEYS, how="left")

def mw(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    x, y = x[~np.isnan(x)], y[~np.isnan(y)]
    r = stats.mannwhitneyu(x, y, alternative="two-sided")
    return r.statistic, r.pvalue

def report(name_a, a, name_b, b, unit=""):
    u, p = mw(a, b)
    a, b = np.asarray(a, float), np.asarray(b, float)
    print(f"{name_a}: n={np.sum(~np.isnan(a))}, median {np.nanmedian(a):.4g}{unit} | "
          f"{name_b}: n={np.sum(~np.isnan(b))}, median {np.nanmedian(b):.4g}{unit} | "
          f"difference of medians {np.nanmedian(a) - np.nanmedian(b):.4g} | Mann-Whitney U={u:.1f}, P={p:.4g}")

def wilcoxon(a, b):
    r = stats.wilcoxon(a, b)
    return r.statistic, r.pvalue

def by_animal(df, col):
    g = df.groupby(["condition", "subject"])[col].agg(["count", "median"]).round(3)
    print(g.to_string())

def group(df, ct, cond):
    return df[(df.cell_type == ct) & (df.condition == cond)]

## Data check

In [ ]:
print("Somatic current-step cells by group (cell type, condition):")
print(fi.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), mice=("subject", "nunique"),
      rheobase_min=("rheobase", "min"), rheobase_max=("rheobase", "max")).to_string())
print()
print("Dendritic line-scan cells by group, and how many have a proximal-distal pair:")
print(den_all.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), with_index=("index", "count"),
      mice=("subject", "nunique"), index_min=("index", "min"), index_max=("index", "max")).round(3).to_string())
print()
print("Somatic sweeps: membrane potential", round(somatic.v_min.min(), 1), "to", round(somatic.v_max.max(), 1), "mV;",
      "sampling", sorted(somatic.fs.round().unique()), "Hz; samples per sweep", sorted(somatic.n_samples.unique()))
print("steps (pA):", sorted(somatic.current_pA.unique()))
print("step window (s from sweep start):", somatic.step_on.min(), "to", somatic.step_off.max(),
      "| step onsets", sorted(somatic.step_on.round(3).unique()), "offsets", sorted(somatic.step_off.round(3).unique()))
print("cells with a repeated step:", int((fi.n_repeated_steps > 0).sum()), "of", len(fi),
      "| cells missing a step between 20 and 300 pA:", int((fi.n_steps_20_300 < 15).sum()),
      "| steps present per cell:", fi.n_steps_20_300.value_counts().to_dict())
print()
print("Clock check, somatic: spike times against the commanded step")
spk = np.concatenate([st - on for st, on in zip(somatic.spike_times, somatic.step_on)])
dur = (somatic.step_off - somatic.step_on).median()
print(f"  {len(spk)} spikes; {np.mean((spk >= 0) & (spk <= dur + 0.01)):.4f} fall within the step (0 to {dur:.3f} s + 10 ms)")
neg = somatic[somatic.current_pA <= 0]
print(f"  spikes in sweeps with zero or negative steps: {int(neg.n_spikes.sum())} in {len(neg)} sweeps")
print()
print("Dendritic trials: line-scan sampling", sorted((1 / dendritic.t.apply(lambda t: np.median(np.diff(t)))).round().unique()),
      "Hz; samples per trial", sorted(dendritic.green.apply(len).unique())[:5], "...",
      "; trial start minus sweep start (s):", dendritic.t_offset.min(), dendritic.t_offset.max())
print("current per dendritic pulse (pA):", sorted(dendritic.current_pA.unique()))
print("green profile range", round(min(dendritic.green.apply(np.min)), 1), "to", round(max(dendritic.green.apply(np.max)), 1),
      "| red", round(min(dendritic.red.apply(np.min)), 1), "to", round(max(dendritic.red.apply(np.max)), 1), "(a.u.)")
print("proximal-distal pairs:", len(den_pairs), "| basal ratio", round(den_pairs.basal.min(), 2), "to", round(den_pairs.basal.max(), 2),
      "| pairs outside 0.5 to 2:", int((~den_pairs.kept).sum()))

In [ ]:
# Clock check, dendritic: the Ca2+ signal in the window after the current injections at 1 s
# against a window of the same length before them (baseline 0.0 to 0.4 s, signal 0.4 to 0.95 s).
def site_area(trials, base, win):
    n = min(len(x) for x in trials.green)
    t = trials.t.iloc[0][:n]
    g = np.mean([lid._smooth(x[:n] - lid.GREEN_BG) for x in trials.green], axis=0)
    r = np.mean([lid._smooth(x[:n] - lid.RED_BG) for x in trials.red], axis=0)
    dt = np.median(np.diff(t))
    b = (t >= base[0]) & (t < base[1]); w = (t >= win[0]) & (t < win[1])
    return np.nansum((g[w] - np.nanmean(g[b])) / np.nanmean(r[b])) * dt

clk = []
for k, g in dendritic.groupby(KEYS + ["site"], sort=False):
    clk.append((site_area(g, (0.5, 0.95), (0.95, 1.5)), site_area(g, (0.05, 0.4), (0.4, 0.95))))
clk = np.array(clk)
print(f"sites: {len(clk)}; median area after the injections {np.median(clk[:, 0]):.4f}, "
      f"before them {np.median(clk[:, 1]):.4f}; positive after: {np.mean(clk[:, 0] > 0):.3f}, before: {np.mean(clk[:, 1] > 0):.3f}")
print(f"Wilcoxon after vs before: P = {stats.wilcoxon(clk[:, 0], clk[:, 1]).pvalue:.3g}")

In [ ]:
print("Sr2+-oEPSC cells by group:")
print(sr.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), mice=("subject", "nunique"),
      sweeps=("n_sweeps", "sum"), noise_sd_min=("noise_sd", "min"), noise_sd_max=("noise_sd", "max")).round(3).to_string())
print("sweeps per cell:", sr.n_sweeps.value_counts().to_dict(), "| sampling (Hz):", sorted(sr.fs.round().unique()),
      "| flash at", sorted(sr.flash_t.round(4).unique()), "s from the sweep start")
print()
print("Clock check, Sr2+: events 40 to 400 ms after the flash against 480 to 840 ms after it (same detector, same length)")
print(f"  median per cell {sr.n_events_5.median():.0f} against {sr.n_events_late.median():.0f}; "
      f"more events early in {np.mean(sr.n_events_5 > sr.n_events_late):.2f} of cells; "
      f"Wilcoxon P = {stats.wilcoxon(sr.n_events_5, sr.n_events_late).pvalue:.3g}")

In [ ]:
print("GRAB-ACh3.0 regions by group (single-pulse trials):")
nod = grab[grab.drug == "control"]
print(nod.groupby("state").agg(regions=("region", "size"), mice=("subject", "nunique"),
      auc_min=("auc", "min"), auc_max=("auc", "max")).round(3).to_string())
print("drugs per group:", grab.groupby("state").drug.unique().apply(list).to_dict())
print("trials per region and drug:", grab.n_trials.value_counts().to_dict())
print("pixel size (um):", sorted(grab_trials.spacing.round(4).unique()), "| pixels in the 50 um circle:", sorted(grab_trials.mask_px.unique()))
fps = grab_trials.t.apply(lambda t: 1 / np.median(np.diff(t)))
print("frame rate (Hz):", sorted(fps.round(2).unique()), "| trials above 10 Hz:", grab_trials[fps > 10][["subject", "region", "stimulation"]].drop_duplicates().to_dict("records"))
print("stimulus delay after trial start (s):", sorted((grab_trials.stim - grab_trials.start).dropna().round(1).unique()))
print("regions whose Fmax comes from another region of the slice:", grab[grab.drug == "control"].fmax_from.value_counts().to_dict())
print("regions with no TTX trial (Fmin from the quinpirole baseline):", int(grab[grab.drug == "control"].fttx.isna().sum()))
print()
print("Clock check, GRAB: dF in the 2 s after the stimulus against the 2 s before it (F0 from 3 to 2 s before), single pulses without drug")
pre, post = [], []
for _, r in grab_trials[(grab_trials.drug == "control") & grab_trials.stimulation.str.startswith("single")].iterrows():
    x, t = r.circle, r.t
    f0 = x[(t >= r.stim - 3) & (t < r.stim - 2)].mean()
    pre.append(np.mean(x[(t >= r.stim - 2) & (t < r.stim)] - f0)); post.append(np.mean(x[(t >= r.stim) & (t < r.stim + 2)] - f0))
pre, post = np.array(pre), np.array(post)
print(f"  trials {len(pre)}; median dF after {np.median(post):.2f}, before {np.median(pre):.2f} (a.u.); after > before in {np.mean(post > pre):.2f}")

In [ ]:
print("AIM sessions by figure and group:")
print(aim.groupby(["figure", "group"]).agg(mice=("mouse", "nunique"), sessions=("path", "nunique")).to_string())
print("time points (min after levodopa):", sorted((aim.time / 60).unique()))
for b in ["axial", "limb", "orolingual", "total"]:
    x = aim[aim.behavior == b].score
    print(f"{b}: {x.min()} to {x.max()}, n={len(x)}")
tl = aim.pivot_table(index=["path", "time"], columns="behavior", values="score")
print("time points where the lab's total differs from axial + limb + orolingual:",
      int((~np.isclose(tl.total, tl.axial + tl.limb + tl.orolingual)).sum()), "of", len(tl))
print("sessions per mouse:", aim.groupby("mouse").session.nunique().value_counts().to_dict())
print("genotype label in the file name against the group in the session description:")
print(aim.groupby(["figure", "group", "genotype"]).mouse.nunique().to_string())

In [ ]:
# Recordings deposited more than once: identical spike-count vectors and voltage extremes across cells,
# identical dendritic site areas, identical Sr2+ event lists, identical GRAB traces, identical AIM sheets.
fp = somatic.groupby(["path", "cell", "condition"]).apply(lambda g: tuple(np.round(g.sort_values("sweep_start")[["n_spikes", "v_min", "v_max"]].to_numpy().ravel(), 3)))
print("somatic cells:", len(fp), "| duplicated fingerprints:", int(fp.duplicated().sum()))
dfp = den_sites.groupby(["path", "cell", "condition"]).area.apply(lambda a: tuple(np.round(sorted(a), 6)))
print("dendritic cells:", len(dfp), "| duplicated site areas:", int(dfp.duplicated().sum()))
sfp = sr.apply(lambda r: (r.n_events_5, round(r.mean_amp_5, 6)), axis=1)
print("Sr2+ cells:", len(sfp), "| duplicated:", int(sfp.duplicated().sum()))
gfp = grab_trials.apply(lambda r: tuple(np.round(r.circle[:20], 3)), axis=1)
print("GRAB trials:", len(gfp), "| duplicated traces:", int(gfp.duplicated().sum()))
afp = aim.groupby("path").score.apply(tuple)
print("AIM sessions:", len(afp), "| identical score sheets:", int(afp.duplicated(keep=False).sum()))
dup = aim.groupby("path").score.apply(tuple).reset_index()
dup = dup[dup.score.duplicated(keep=False)]
if len(dup):
    print(dup.merge(aim[["path", "mouse", "session"]].drop_duplicates(), on="path")[["mouse", "session"]].to_string())

In [ ]:
# The paper's n (figure legends) next to the dandiset's counts. The legend numbers are the paper's, typed here
# as reference values; every dandiset count is computed above.
legend = pd.DataFrame([
    ("somatic", "dSPN", "LID off-state", 15, 8), ("somatic", "dSPN", "LID on-state", 15, 10),
    ("somatic", "dSPN", "LID on-state with SCH 23390", 10, 5), ("somatic", "iSPN", "LID off-state", 12, 5),
    ("somatic", "iSPN", "LID on-state", 11, 8), ("somatic", "iSPN", "LID on-state with sulpiride", 7, 4),
    ("somatic", "iSPN", "LID off-state, vehicle control", 8, 4), ("somatic", "iSPN", "LID off-state with M1R antagonists", 10, 5),
    ("somatic", "iSPN", "CDGI knockout, LID off-state", 13, 5), ("somatic", "iSPN", "CDGI knockout, LID on-state", 8, 4),
    ("somatic", "iSPN", "LID off-state, control (gRNA only)", 6, 3), ("somatic", "iSPN", "LID off-state, M1R deletion", 8, 4),
    ("dendritic", "dSPN", "LID off-state", 10, 7), ("dendritic", "dSPN", "LID on-state", 10, 5),
    ("dendritic", "dSPN", "LID on-state with SCH 23390", 9, 5), ("dendritic", "iSPN", "LID off-state", 10, 5),
    ("dendritic", "iSPN", "LID on-state", 10, 5), ("dendritic", "iSPN", "LID on-state with sulpiride", 8, 6),
    ("dendritic", "iSPN", "LID off-state, vehicle control", 9, 5), ("dendritic", "iSPN", "LID off-state with M1R antagonists", 7, 4),
    ("dendritic", "iSPN", "CDGI knockout, LID off-state", 11, 5), ("dendritic", "iSPN", "CDGI knockout, LID on-state", 8, 4),
    ("Sr2+", "dSPN", "LID off-state", 10, 4), ("Sr2+", "dSPN", "LID on-state", 8, 4),
    ("Sr2+", "iSPN", "LID off-state", 7, 4), ("Sr2+", "iSPN", "LID on-state", 8, 4)],
    columns=["assay", "cell_type", "condition", "paper_cells", "paper_mice"])
cnt = pd.concat([
    fi.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), mice=("subject", "nunique")).assign(assay="somatic"),
    den_all.dropna(subset=["index"]).groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), mice=("subject", "nunique")).assign(assay="dendritic"),
    sr.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), mice=("subject", "nunique")).assign(assay="Sr2+")]).reset_index()
cmp_ = legend.merge(cnt, on=["assay", "cell_type", "condition"], how="left")
print(cmp_.to_string())
print("groups whose cell count differs:", int((cmp_.cells != cmp_.paper_cells).sum()), "| whose mouse count differs:", int((cmp_.mice != cmp_.paper_mice).sum()))

In [ ]:
# The files' condition column against the session name (cell type, state, drug, genotype) and the subject folder's tag.
# The claims file groups cells by a condition_subfolder column, which these files do not have; their condition
# column is used. Each condition should fall in one kind of folder.
som_c = fi.assign(name_state=fi.path.str.split("ses-").str[1].str.split("\\+\\+").str[1],
                  name_drug=fi.path.str.split("ses-").str[1].str.split("\\+\\+").str[2],
                  name_genotype=fi.path.str.split("ses-").str[1].str.split("\\+\\+").str[3],
                  folder_tag=fi.subject.str.extract(r"Subject\d{8}([A-Za-z0-9]+?)(?:eGFP|tDTomato|Adora2Cre|CDGIkoX)")[0])
print("Somatic cells: condition against session name and folder tag")
print(som_c.groupby(["cell_type", "condition", "name_state", "name_drug", "name_genotype", "folder_tag"]).size().to_string())
den_c = den_all.assign(name=den_all.path.str.split("ses-").str[1].str.rsplit("++", n=1).str[0],
                       folder_tag=den_all.subject.str.extract(r"Subject\d{8}([A-Za-z0-9]+?)(?:eGFP|tDTomato|Adora2Cre|CDGIkoX)")[0])
print("\nDendritic cells: condition against session name and folder tag")
print(den_c.groupby(["cell_type", "condition", "name", "folder_tag"]).size().to_string())
print("\nicephys files (no imaging) by the start of their session description, and whether they are read as Sr2+-oEPSC files:")
print(pd.DataFrame({"description": ic_desc.str[:45], "Sr2+ file": ic_desc.index.isin(sr_files.index)}).value_counts().to_string())


In [ ]:
# Image files: the files without an acquisition suffix in the patch-clamp cell types, by their session description.
img = assets[(assets.kind == "") & assets.cell_type.isin(["dSPN", "iSPN"])].copy()
img["description"] = [lid.session_description(x) for x in img.asset_id]
img["what"] = np.select([img.description.str.startswith("Dendritic spine density"), img.description.str.startswith("High-resolution confocal"),
                         img.description.str.startswith("Overview")], ["2PLSM spine Z-stacks", "confocal Z-stacks", "confocal overview"], "other")
print(img.groupby(["what", "cell_type", "state", "genotype"]).size().to_string())
print("\nconfocal files that name Figure 4, J and K:", int(img.description.str.contains("Figure 4, J and K").sum()), "of", int((img.what == "confocal Z-stacks").sum()))
print("example 2PLSM description:", img[img.what == "2PLSM spine Z-stacks"].description.iloc[0][:330])
print("example confocal description:", img[img.what == "confocal Z-stacks"].description.iloc[0][:250])
print("overview:", img[img.what == "confocal overview"].description.iloc[0][:200])
print("files with a spine count or spine table:", int(img.description.str.contains("spines per|spine count", case=False).sum()))


The counts match the paper's legends for every somatic, dendritic, Sr2+ and GRAB group, with two exceptions: two cells with dendritic line scans have no index (below). Mouse counts differ in four groups (Sr2+ dSPN and iSPN off-state, 3 subjects against the legends' 4; M1R CRISPR control and deletion, 2 and 3 against 3 and 4). The dandiset has one subject per recording day, so mice recorded on one day would share a subject; the files cannot tell.

Checks of the description against the data:
- Membrane potential is in volts with the stored conversion (-125 to 53 mV), steps run from 0.2 to 0.7 s of each sweep, as the 500 ms steps of the paper. Every one of the 11886 spikes falls inside the commanded step, and no sweep with a zero or negative step has a spike, so the voltage and the stimulus share a clock.
- The line scans start with their sweeps (offset 0), the dendritic pulses are 2 nA as the Methods give, and the Ca2+ signal after the injections at 1 s is positive at every one of 425 sites, against 3.5% for a window of the same length before them (Wilcoxon P = 2e-71).
- Sr2+ sweeps have the flash at 20 ms, as the series description states; the detector finds a median 89 events per cell in the 40 to 400 ms window and 14 in a window of the same length 480 to 840 ms after the flash, more early in every cell.
- GRAB-ACh3.0 pixels are 0.388 um, as the Methods give, so the 50 um circle covers nearly the whole 128 x 128 field. The stimulus comes 2.9 s or 9.8 to 9.9 s after the trial start (10 s or 3 s, as the trials table describes). After a single pulse the fluorescence rises in every one of 74 trials without drug. Two burst trials of one region were filmed at 21.26 Hz on a 64 x 64 field; they are not single pulses and are not used. One region has no acetylcholine trial (its Fmax is taken from the other region of its slice) and one has no TTX trial (Fmin from the quinpirole baseline, as the Methods allow).
- AIM scores run from 0 to 4, the lab's total from 0 to 12, at 20 to 120 min after levodopa; the total differs from the sum of the three scores at 5 of 1064 time points. The genotype in each file name agrees with the group in its session description.
- Each condition used to group cells falls in files of one state, genotype and folder tag (printed above). The on-state groups also take the baseline cells of the files named for SCH 23390 (6 dSPNs) and sulpiride (5 somatic and 1 dendritic iSPN), as the claims file expects ("files named for a drug also hold baseline cells"), and the SCH 23390 and sulpiride groups come only from those files. So the condition column agrees with the session names and folders. The Sr2+-oEPSC files are the 14 icephys files whose session description begins "Sr2+-oEPSC recordings"; the other 8 describe current-step patch-clamp recordings (Figures 1, 6 and 9).
- The dandiset holds 76 two-photon spine Z-stack files and 16 confocal Z-stack files (all naming Figure 4, J and K) with one overview image of Figure 4I; no description mentions a spine count.
- No somatic, dendritic, Sr2+ or GRAB recording is deposited twice. Two AIM score sheets are identical: mouse 7830, sessions 4 and 5 (the paper's table has the same values for both).
- In 18 of 123 somatic cells a step between 20 and 300 pA is missing and in 14 one is recorded twice, as the dandiset description says; the F-I sums run over the steps present, and a repeated step keeps its first sweep (its last sweep is shown as a variant).

## link-somatic-cells

> The dandiset's somatic current-step recordings are those behind the paper's somatic excitability figures, and the spike counts computed from them agree with the paper's per-cell tables.

Criterion (claims file): groups matched by name; each dandiset cell compared with every row of its group over the steps from 20 to 300 pA present in both; best match the row with the smallest summed absolute difference. Reproduced when the cell counts per group equal the paper's n, at least 80% of cells match exactly and at least 95% within 3 spikes; partially reproduced when at least 50% match exactly. Groups come from the file's `condition` column (the claims file names `condition_subfolder`, which this version does not have).

In [ ]:
SOM = {"Fig 1E": ("Fig 1", "Fig 1E_dSPN somatic excitabilit", "dSPN",
                  {"LID off": "LID off-state", "LID on": "LID on-state", "LID on + SCH": "LID on-state with SCH 23390"}),
       "Fig 3C": ("Fig 3", "Fig 3C_iSPN somatic excitabilit", "iSPN",
                  {"LID off": "LID off-state", "LID on": "LID on-state", "LID on + sul": "LID on-state with sulpiride"}),
       "Fig 6C": ("Fig 6", "Fig 6C_M1R antagonist SE", "iSPN",
                  {"control": "LID off-state, vehicle control", "M1R antagonist": "LID off-state with M1R antagonists"}),
       "Fig 7C": ("Fig 7", "Fig 7C_KO somatic excitability", "iSPN",
                  {"KO off": "CDGI knockout, LID off-state", "KO on": "CDGI knockout, LID on-state"}),
       "Fig 9D": ("Fig 9", "Fig 9D_M1R CRISPR SE", "iSPN",
                  {"control": "LID off-state, control (gRNA only)", '"M1R"': "LID off-state, M1R deletion"})}
steps = lid.FI_STEPS
rows = []
for fig, (book, sheet, ct, gmap) in SOM.items():
    t = tables[book][sheet].copy()
    t.columns = ["id", "group"] + [int(c.split(" pA")[0]) for c in t.columns[2:]]
    for pg, dg in gmap.items():
        P = t[t.group == pg].reset_index(drop=True)
        cells = group(fi, ct, dg).sort_values(["date", "cell"])
        for order, (_, c) in enumerate(cells.iterrows()):
            v = np.array([c[f"s{s}"] for s in steps]); ok = ~np.isnan(v)
            diffs = np.array([np.abs(v[ok] - r[steps].to_numpy(float)[ok]).sum() for _, r in P.iterrows()])
            j = int(np.argmin(diffs)); pr = P.iloc[j]
            pos = [s for s in steps if pr[s] > 0]
            rows.append(dict(fig=fig, group=pg, date=c.date, cell=c.cell, order=order, best_row=j, best_diff=diffs[j],
                             n_tied=int((diffs == diffs[j]).sum()), rheobase=c.rheobase,
                             paper_first_step_with_spike=min(pos) if pos else np.nan, n_paper=len(P), n_dandi=len(cells)))
link_som = pd.DataFrame(rows)
summ = link_som.groupby(["fig", "group"]).agg(n_paper=("n_paper", "first"), n_dandiset=("n_dandi", "first"),
        exact=("best_diff", lambda x: (x == 0).mean()), within3=("best_diff", lambda x: (x <= 3).mean())).round(3)
print(summ.to_string())
n_equal = int((summ.n_paper == summ.n_dandiset).sum())
exact, within3 = (link_som.best_diff == 0).mean(), (link_som.best_diff <= 3).mean()
print(f"\ngroups with the paper's n: {n_equal} of {len(summ)}; cells {len(link_som)}; exact match {exact:.3f}; within 3 spikes {within3:.3f}")
print("cells whose best match is tied between rows:", int((link_som.n_tied > 1).sum()))
m = link_som[link_som.best_diff == 0]
print(f"rheobase equal to the first step with a spike in the matched row: {np.mean(m.rheobase == m.paper_first_step_with_spike):.3f} of exact matches; "
      f"{np.mean(link_som.rheobase == link_som.paper_first_step_with_spike):.3f} of all cells")
print("exact matches whose matched row is the cell's position in date and cell order:", f"{np.mean(m.best_row == m.order):.3f}")
print("\ncells without an exact match:")
print(link_som[link_som.best_diff > 0][["fig", "group", "date", "cell", "best_diff", "rheobase", "paper_first_step_with_spike"]].to_string())

**Result: partially reproduced.** Every one of the 12 groups holds the paper's n (123 cells). 73.2% of cells match a row of the paper's table exactly over 20 to 300 pA and 85.4% within 3 spikes, below the 80% and 95% of the criterion and above its 50% for a partial link. 97.8% of the exact matches fall on the row at the cell's position in date and cell order, so the paper's rows follow that order. The misses are largest in dSPN on-state (40% exact) and on-state with sulpiride (57%); the largest differences are 20 and 24 spikes in total.

## link-dendritic-cells

> The dandiset's dendritic line scans are those behind the paper's dendritic excitability index figures, and the index computed from them agrees with the paper's per-cell values.

Criterion (claims file): Figure 6D rows matched by date and letter (a, b, c for the first, second and third cell of the day, read here as cell 1, 2, 3, which is how the 6D identifiers line up with the file's cell numbers, e.g. 02202020_c is cell 3); elsewhere rows taken in the dandiset's order by date and cell number. Every cell with line scans keeps its place in that order, whether or not it has a proximal-distal pair. Reproduced when the counts of cells with an index equal the paper's n in at least 7 of the 10 groups and the matched values differ by a median of at most 10% of the paper's value; partially reproduced when only one holds.

In [ ]:
DEN = {"Fig 1I": ("Fig 1", "Fig 1I_dSPN dendritic excitabil", "dSPN",
                  {"LID off": "LID off-state", "LID on": "LID on-state", "LID on + SCH": "LID on-state with SCH 23390"}),
       "Fig 3F": ("Fig 3", "Fig 3F_iSPN dendritic excitabil", "iSPN",
                  {"LID off": "LID off-state", "LID on": "LID on-state", "LID on + sul": "LID on-state with sulpiride"}),
       "Fig 6D": ("Fig 6", "Fig 6D_M1R antagonist DE", "iSPN",
                  {"control": "LID off-state, vehicle control", "M1R antagonist": "LID off-state with M1R antagonists"}),
       "Fig 7F": ("Fig 7", "Fig 7F_KO dendritic excitabilit", "iSPN",
                  {"KO|LID off": "CDGI knockout, LID off-state", "KO|LID on": "CDGI knockout, LID on-state"})}
rows, matched = [], []
for fig, (book, sheet, ct, gmap) in DEN.items():
    t = tables[book][sheet].copy()
    if fig == "Fig 7F":
        t["group"] = t.iloc[:, 1] + "|" + t.iloc[:, 2]; t["paper"] = t.iloc[:, 3]
    else:
        t["group"] = t.iloc[:, 1]; t["paper"] = t.iloc[:, 2]
    for pg, dg in gmap.items():
        P = t[t.group == pg].reset_index(drop=True)
        D = group(den_all, ct, dg).sort_values(["date", "cell"]).reset_index(drop=True)
        if fig == "Fig 6D":
            P["date"] = P.iloc[:, 0].str.split("_").str[0].apply(lambda s: s[4:] + s[:4])
            P["cell"] = P.iloc[:, 0].str.split("_").str[1].apply(lambda c: ord(c) - 96)
            M = D.merge(P[["date", "cell", "paper"]], on=["date", "cell"], how="outer")
        else:
            n = min(len(P), len(D)); M = D.iloc[:n].copy(); M["paper"] = P.paper.iloc[:n].to_numpy()
        M["fig"], M["group"] = fig, pg
        M["rel_diff"] = (M["index"] - M.paper).abs() / M.paper
        matched.append(M)
        rows.append(dict(fig=fig, group=pg, n_paper=len(P), n_with_index=int(D["index"].notna().sum()),
                         n_line_scan_cells=len(D), n_matched=int(M.rel_diff.notna().sum()),
                         median_rel_diff=M.rel_diff.median(), within_10pct=(M.rel_diff <= 0.10).mean()))
link_den = pd.DataFrame(rows)
link_den_cells = pd.concat(matched, ignore_index=True)
print(link_den.round(3).to_string())
n_eq = int((link_den.n_paper == link_den.n_with_index).sum())
med = link_den_cells.rel_diff.median()
print(f"\ngroups whose count of cells with an index equals the paper's n: {n_eq} of {len(link_den)}")
print(f"median relative difference over all {int(link_den_cells.rel_diff.notna().sum())} matched cells: {med:.3f}; "
      f"within 10%: {(link_den_cells.rel_diff <= 0.10).mean():.3f}")
print("\ncells with line scans but no proximal-distal pair:")
print(den_all[den_all["index"].isna()][["date", "cell", "condition"]].to_string())
print(den_sites.merge(den_all[den_all["index"].isna()][KEYS], on=KEYS)[["date", "cell", "site"]].to_string())

**Result: reproduced.** The count of cells with an index equals the paper's n in 8 of 10 groups (iSPN on-state and CDGI KO off-state have one cell fewer, each a cell imaged at one dendritic site only, at two spots). Over the 90 matched cells the index differs from the paper's by a median of 2.8% of its value, and 83.7% are within 10%. The Figure 6D identifiers line up with the files' cell numbers (median difference 3.5% and 6.4% in its two groups).

## link-sr-cells

> The dandiset's Sr2+-oEPSC recordings are cells of the paper's Figures 2 and 4, and their events agree with the paper's per-cell values.

Criterion (claims file): cells matched by date and session letter; reproduced when every dandiset cell matches a paper identifier and the per-cell mean amplitude and event count are within 25% of the paper's for at least 75% of cells; partially reproduced when every cell matches but the values do not. The letter is read as the cell number (a = 1, c = 3), as for Figure 6D. The paper's event count is its frequency times 3.6 s (10 sweeps of 360 ms). Events come from `lid.detect_events` at 5 SD (Methods); 3 and 4 SD and the per-cell median are shown as variants.

In [ ]:
pa = pd.concat([
    tables["Fig 2"]["Fig 2H_dSPN Sr-oEPSC amp"].set_axis(["id", "group", "amp"], axis=1)
        .merge(tables["Fig 2"]["Fig 2I_dSPN Sr-oEPSC freq"].set_axis(["id", "group", "freq"], axis=1)),
    tables["Fig 4"]["Fig 4G_iSPN Sr-oEPSC amp"].set_axis(["id", "group", "amp"], axis=1)
        .merge(tables["Fig 4"]["Fig 4H_iSPN Sr-oEPSC freq"].set_axis(["id", "group", "freq"], axis=1))])
pa["n_events_paper"] = pa.freq * 3.6
sr["id"] = sr.apply(lambda r: (r.date[4:8] + r.date[:4] if r.cell_type == "dSPN" else r.date) + chr(96 + r.cell), axis=1)
lk = sr.merge(pa, on="id", how="outer", indicator=True)
print("matched:", lk._merge.value_counts().to_dict())
for n in (5, 4, 3):
    ra = (lk[f"mean_amp_{n}"] - lk.amp).abs() / lk.amp
    rn = (lk[f"n_events_{n}"] - lk.n_events_paper).abs() / lk.n_events_paper
    rm = (lk[f"median_amp_{n}"] - lk.amp).abs() / lk.amp
    print(f"{n} SD: mean amplitude within 25%: {np.mean(ra <= .25):.3f}; event count within 25%: {np.mean(rn <= .25):.3f}; "
          f"both: {np.mean((ra <= .25) & (rn <= .25)):.3f} | median amplitude within 25%: {np.mean(rm <= .25):.3f}, "
          f"with the count: {np.mean((rm <= .25) & (rn <= .25)):.3f} | "
          f"median ratio of counts (dandiset/paper) {np.median(lk[f'n_events_{n}'] / lk.n_events_paper):.2f}")
print(lk[["id", "group", "n_sweeps", "amp", "mean_amp_5", "median_amp_5", "n_events_paper", "n_events_5"]].round(2).to_string())
# The paper's per-cell "average" amplitude against the mean and the median of its own event lists (Figures 2G and 4F)
ev = pd.concat([tables["Fig 2"]["Fig 2G_dSPN Sr-oEPSC histogram"].set_axis(["id", "group", "ev", "a"], axis=1),
                tables["Fig 4"]["Fig 4F_iSPN Sr-oEPSC histogram"].set_axis(["id", "group", "ev", "a"], axis=1)])
evs = ev.groupby("id").a.agg(["mean", "median", "size"]).join(pa.set_index("id")[["amp", "n_events_paper"]])
print("\npaper's per-cell amplitude equal (within 0.01 pA) to the mean of its event list:", int(np.isclose(evs["mean"], evs.amp, atol=0.01).sum()),
      "| to the median:", int(np.isclose(evs["median"], evs.amp, atol=0.01).sum()), "of", len(evs))
print("event list length equal to frequency x 3.6 s:", int(np.isclose(evs["size"], evs.n_events_paper, atol=0.01).sum()), "of", len(evs))

**Result: partially reproduced.** All 33 dandiset cells match a paper identifier, and all 33 paper cells are in the dandiset. At 5 SD the mean amplitude is within 25% of the paper's value for 90.9% of cells, but the event count for 42.4%, and both for 39.4%, below the criterion's 75%. The stand-in detector finds a median 1.33 times the paper's events (1.50 at 4 SD, 2.14 at 3 SD). The paper's per-cell "average" amplitude equals the median of its own event list for all 33 cells and its mean for none; the event lists hold exactly frequency x 3.6 s events, 10 sweeps of 360 ms, while two dandiset cells hold 11 and 12 sweeps. The criterion compares the per-cell mean amplitude; since the paper's value is a median, this attempt asks a reader whether the criterion should compare the median (questions/zhai-2025.yaml). With the median, 66.7% of cells have the amplitude within 25% and 42.4% both amplitude and count, below 75% too, so the outcome is the same under either reading.

## link-grab-rois

> The dandiset's GRAB-ACh recordings are regions of the paper's Figure 5F, and the normalised areas computed from them agree with the paper's per-region values.

Criterion (claims file): each mouse's vector of single-pulse areas (no drug, quinpirole, sulpiride, and dopamine where present) matched to the paper's region in the same group with the smallest summed absolute difference; reproduced when the counts per group are those of the paper and every mouse has a match within 0.05 per treatment; partially reproduced when the dandiset holds part of the regions and at least half match within 0.05 per treatment. The criterion was written for one region per mouse; this version holds several regions per mouse, so the matching is done per region, the unit of the paper's table.

In [ ]:
pg = tables["Fig 5"]["Fig 5F_GRAB-ACh_single-AUC"].set_axis(["roi", "group", "drug", "auc"], axis=1)
dmap = {"No drug": "control", "quinpirole": "10 µM quinpirole", "sulpiride": "10 µM sulpiride", "DA": "50 nM dopamine"}
gmap = {"unlesioned": "UnlesionedControl", "6-OHDA": "LesionedControl", "LID off": "OffState"}
pg["drug"] = pg.drug.map(dmap); pg["state"] = pg.group.map(gmap)
PV = pg.pivot_table(index=["state", "roi"], columns="drug", values="auc")
DV = grab.pivot_table(index=["state", "subject", "region"], columns="drug", values="auc")
rows = []
for (state, subj, region), v in DV.iterrows():
    cand = PV.loc[state]
    cols = [c for c in v.index if not np.isnan(v[c]) and c in cand.columns and cand[c].notna().any()]
    d = (cand[cols] - v[cols]).abs()
    j = d.sum(axis=1).idxmin()
    rows.append(dict(state=state, subject=subj, region=region, paper_roi=j, sum_abs=d.loc[j].sum(), max_abs=d.loc[j].max()))
lg = pd.DataFrame(rows)
cnt = pd.DataFrame({"paper": PV.groupby(level=0).size(), "dandiset": DV.groupby(level=0).size()})
print(cnt.to_string())
print(f"\nregions matched within 0.05 for every treatment: {int((lg.max_abs <= 0.05).sum())} of {len(lg)}; "
      f"median of the largest per-treatment difference {lg.max_abs.median():.3f}")
print("paper regions chosen by more than one dandiset region:", int(lg.duplicated(["state", "paper_roi"]).sum()))
print(lg.round(3).to_string())
# Variants: the Fmin rule (the Methods' min(TTX, quinpirole), TTX only as the script, the quinpirole baseline only),
# each matched many-to-one as above and one-to-one (Hungarian assignment on the summed difference, within each group).
from scipy.optimize import linear_sum_assignment
def grab_link(gdf):
    DVv = gdf.pivot_table(index=["state", "subject", "region"], columns="drug", values="auc")
    out = []
    for state in PV.index.get_level_values(0).unique():
        cand, D = PV.loc[state], DVv.loc[state]
        S = np.zeros((len(D), len(cand))); M = np.zeros_like(S)
        for i, (_, v) in enumerate(D.iterrows()):
            cols = [c for c in v.index if not np.isnan(v[c]) and c in cand.columns and cand[c].notna().any()]
            d = (cand[cols] - v[cols]).abs()
            S[i], M[i] = d.sum(axis=1).to_numpy(), d.max(axis=1).to_numpy()
        r, c = linear_sum_assignment(S)
        oto = dict(zip(r, c))
        for i in range(len(D)):
            b = int(S[i].argmin())
            out.append(dict(state=state, best=cand.index[b], best_max=M[i, b], oto_max=M[i, oto[i]] if i in oto else np.nan))
    return pd.DataFrame(out)
rows = []
for rule in ["min", "ttx", "quin"]:
    L = grab_link(lid.grab_auc(grab_trials, fmin_rule=rule))
    g0 = lid.grab_auc(grab_trials, fmin_rule=rule)
    med = g0[g0.drug == "control"].groupby("state").auc.median()
    rows.append(dict(fmin=rule, regions=len(L), many_to_one_within_005=int((L.best_max <= .05).sum()),
                     paper_regions_reused=int(L.duplicated(["state", "best"]).sum()), one_to_one_within_005=int((L.oto_max <= .05).sum()),
                     median_max_diff_many=L.best_max.median(), median_max_diff_one=L.oto_max.median(),
                     **{f"median_{k}": med.get(k) for k in ["UnlesionedControl", "LesionedControl", "OffState"]}))
print("\nFmin rules and matching:")
print(pd.DataFrame(rows).round(3).to_string())
print("paper's medians (no drug):", pg[pg.drug == "control"].groupby("state").auc.median().round(3).to_dict())


**Result: not reproduced.** The counts per group are the paper's (13, 10 and 11 regions), but 16 of 34 regions match a paper region within 0.05 for every treatment, short of half, and 10 paper regions are the best match of more than one dandiset region. The median of the largest per-treatment difference is 0.052. The regions cannot be matched one by one (no identifier), the paper's background images are not in the dandiset (they cancel in the normalised area), and the ROI here is the Methods' 50 um circle, not the region Prairie View saved; any of these may explain the gaps. The group comparisons below do not depend on the match. Under the other Fmin rules and with a one-to-one match the agreement is lower or similar (printed above): the Methods' rule, min(TTX, quinpirole), matches the paper's values best, and with it a one-to-one match pairs 15 of 34 regions within 0.05, so the outcome is the same under every reading.

## link-aim-mice

> The dandiset's AIM score tables are mice of the paper's Figures 8 and 10, with genotypes and scores that agree with the paper's tables.

Criterion (claims file): mice matched by numeric identifier; the paper's total AIM score per session is the sum over the session's time points of the three scores. Reproduced when every dandiset mouse matches a paper mouse and every matched session total is equal; partially reproduced when the matched mice agree but some dandiset mice are not in the paper or the genotype labels disagree; not reproduced when matched values differ.

In [ ]:
aim_tot = aim[aim.behavior.isin(["axial", "limb", "orolingual"])].groupby(["figure", "group", "genotype", "mouse", "session"]).score.sum().reset_index()
p8 = tables["Fig 8"]["Fig 8B_total AIMs"].set_axis(["mouse", "pgroup", "session", "paper_total"], axis=1)
p8["mouse"] = p8.mouse.str.extract(r"(\d+)")[0].astype(int)
p10 = tables["Fig 10"]["Fig 10B_total AIMs"].set_axis(["mouse", "pgroup", "session", "paper_total"], axis=1)
P = pd.concat([p8.assign(pfig=8), p10.assign(pfig=10)])
m8, m10 = set(aim_tot[aim_tot.figure == 8].mouse), set(aim_tot[aim_tot.figure == 10].mouse)
print("mouse IDs in both figures, dandiset:", sorted(m8 & m10), "| paper:", sorted(set(p8.mouse) & set(p10.mouse)))
la = aim_tot.merge(P.rename(columns={"pfig": "figure"}), on=["figure", "mouse", "session"], how="left")
la["equal"] = np.isclose(la.score, la.paper_total)
print(la.groupby(["figure", "group"]).agg(mice=("mouse", "nunique"), in_paper=("paper_total", lambda x: x.notna().mean()),
      sessions_equal=("equal", "mean"), paper_group=("pgroup", lambda x: sorted(set(x.dropna())))).round(3).to_string())
print("\ndandiset mice not in the paper's tables:", sorted(la[la.paper_total.isna()].mouse.unique()))
print("paper mice not in the dandiset:", sorted(set(P.mouse) - set(aim_tot.mouse)))
print("\nmouse 7830, sessions 4 and 5 (identical sheets in the dandiset):")
print(la[(la.mouse == 7830) & la.session.isin([4, 5])][["session", "score", "paper_total"]].to_string())
print("\nmatched sessions that differ:")
print(la[la.paper_total.notna() & ~la.equal].pivot_table(index=["figure", "mouse"], columns="session", values=["score", "paper_total"]).to_string())

**Result: not reproduced.** 18 of the 21 Figure 8 mice in the dandiset are in the paper's table (4123, 4124 and 5871 are not; 2346, 2347, 7741, 7742 and 7744 of the paper are not in the dandiset), and their session totals equal the paper's in 6% (control) and 16% (CDGI KO) of sessions. The genotype labels agree. For Figure 10 all 21 mice match, with their groups, and 98% of sessions are equal: two sessions differ (mouse 4790 session 3, 44 against 43; mouse 5845 session 4, 39.5 against 36.5). The link holds for the Figure 10 mice and fails for the Figure 8 mice, as the dandiset description states. No mouse number appears in both figures, in the dandiset or in the paper's tables, and sessions are matched on figure, mouse and session.

## fig1f-dspn-rheobase-on-lower

> In dSPNs the rheobase is lower in the LID on-state than in the off-state.

Criterion: two-sided Mann-Whitney test of per-cell rheobase, on-state against off-state; reproduced when P < 0.05 and the on-state median is lower. Rheobase is the first positive step with a spike, spikes counted as upward crossings of 0 mV (the paper's script, `lid.cell_fi`).

In [ ]:
on, off = group(fi, "dSPN", "LID on-state"), group(fi, "dSPN", "LID off-state")
report("on-state", on.rheobase, "off-state", off.rheobase, " pA")
print("rheobase values (pA), on:", sorted(on.rheobase.tolist()), "| off:", sorted(off.rheobase.tolist()))
by_animal(pd.concat([on, off]), "rheobase")
# variant: a step recorded twice keeps its last sweep
report("on-state (last sweep of a repeated step)", group(fi_last, "dSPN", "LID on-state").rheobase,
       "off-state", group(fi_last, "dSPN", "LID off-state").rheobase, " pA")

**Result: reproduced.** On-state dSPN rheobase median 140 pA (15 cells, 10 mice) against 220 pA off-state (15 cells, 8 mice), Mann-Whitney P = 8.9e-06, as in the paper (P < 0.0001). Keeping the last sweep of a repeated step gives the same.

## fig1e-dspn-fi-on-higher

> In on-state dSPNs suprathreshold current steps evoke more spikes than in off-state dSPNs across a range of intensities.

Criterion: per cell, the sum of spike counts from 20 to 300 pA (over the steps present); reproduced when a two-sided Mann-Whitney test gives P < 0.05 with the on-state higher; the per-step medians are reported.

In [ ]:
report("on-state", on.fi_sum, "off-state", off.fi_sum, " spikes")
med = pd.DataFrame({"on": on[[f"s{s}" for s in steps]].median(), "off": off[[f"s{s}" for s in steps]].median()})
med["P"] = [mw(on[f"s{s}"], off[f"s{s}"])[1] for s in steps]
print(med.round(4).T.to_string())
print("cells with fewer than 15 steps between 20 and 300 pA, on/off:", int((on.n_steps_20_300 < 15).sum()), int((off.n_steps_20_300 < 15).sum()))
# variant: mean spikes per step present, for cells missing steps
report("on-state, mean per step", on.fi_sum / on.n_steps_20_300, "off-state", off.fi_sum / off.n_steps_20_300)
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(steps, med.on, "o-", label="on-state"); ax.plot(steps, med.off, "s-", label="off-state")
ax.set_xlabel("step (pA)"); ax.set_ylabel("median spikes"); ax.legend(); plt.tight_layout(); plt.show()
print("plotted medians range", med[["on", "off"]].min().min(), "to", med[["on", "off"]].max().max(), "; all points inside the axes")
# variant: a step recorded twice keeps its last sweep
report("on-state (last sweep)", group(fi_last, "dSPN", "LID on-state").fi_sum, "off-state (last sweep)", group(fi_last, "dSPN", "LID off-state").fi_sum, " spikes")


**Result: reproduced.** The 20 to 300 pA spike sum is a median 88 on-state against 36 off-state, P = 8.8e-05; per step, on-state medians are higher from 120 pA up, with P < 0.05 at every step from 100 to 300 pA. The mean per step present gives P = 1.4e-04. Keeping the last sweep of a repeated step gives the same (P = 8.8e-05).

## fig1f-dspn-sch-no-change

> Bath application of the D1R antagonist SCH 23390 does not reverse the lower rheobase of on-state dSPNs.

Criterion: a claim of absence; two-sided Mann-Whitney test of rheobase, on-state against on-state with SCH; reproduced when P >= 0.05. A Wilcoxon test on the cells recorded in both conditions is a variant.

In [ ]:
sch = group(fi, "dSPN", "LID on-state with SCH 23390")
report("on-state", on.rheobase, "on-state + SCH", sch.rheobase, " pA")
pair = on.merge(sch, on=["path", "cell"], suffixes=("", "_sch"))
print("cells recorded in both:", len(pair), "| rheobase before", pair.rheobase.tolist(), "after", pair.rheobase_sch.tolist())
if len(pair) >= 2 and (pair.rheobase != pair.rheobase_sch).any():
    print("Wilcoxon (paired) P =", round(wilcoxon(pair.rheobase, pair.rheobase_sch)[1], 4))
by_animal(pd.concat([on, sch]), "rheobase")
# variant: a step recorded twice keeps its last sweep
report("on-state (last sweep)", group(fi_last, "dSPN", "LID on-state").rheobase, "on-state + SCH (last sweep)", group(fi_last, "dSPN", "LID on-state with SCH 23390").rheobase, " pA")


**Result: reproduced.** On-state rheobase median 140 pA (15 cells) and with SCH 23390 140 pA (10 cells), Mann-Whitney P = 1.0 (paper P = 0.589). The 5 cells recorded before and after SCH give Wilcoxon P = 1.0. Keeping the last sweep of a repeated step gives the same (P = 1.0).

## fig3d-ispn-rheobase-on-higher

> In iSPNs the rheobase is higher in the on-state than in the off-state.

Criterion: two-sided Mann-Whitney test of per-cell rheobase; reproduced when P < 0.05 and the on-state median is higher.

In [ ]:
ion, ioff = group(fi, "iSPN", "LID on-state"), group(fi, "iSPN", "LID off-state")
report("on-state", ion.rheobase, "off-state", ioff.rheobase, " pA")
print("rheobase values (pA), on:", sorted(ion.rheobase.tolist()), "| off:", sorted(ioff.rheobase.tolist()))
by_animal(pd.concat([ion, ioff]), "rheobase")
report("on-state (last sweep of a repeated step)", group(fi_last, "iSPN", "LID on-state").rheobase,
       "off-state", group(fi_last, "iSPN", "LID off-state").rheobase, " pA")


**Result: reproduced.** On-state iSPN rheobase median 160 pA (11 cells) against 120 pA off-state (12 cells), P = 0.00083 (paper P = 0.0013). Keeping the last sweep of a repeated step gives the same.

## fig3c-ispn-fi-on-lower

> The F-I relationship of iSPNs shifts to the right in the on-state.

Criterion: per cell, the sum of spike counts from 20 to 300 pA; reproduced when a two-sided Mann-Whitney test gives P < 0.05 with the on-state lower.

In [ ]:
report("on-state", ion.fi_sum, "off-state", ioff.fi_sum, " spikes")
med = pd.DataFrame({"on": ion[[f"s{s}" for s in steps]].median(), "off": ioff[[f"s{s}" for s in steps]].median()})
print(med.T.to_string())
report("on-state, mean per step", ion.fi_sum / ion.n_steps_20_300, "off-state", ioff.fi_sum / ioff.n_steps_20_300)
# variant: a step recorded twice keeps its last sweep
report("on-state (last sweep)", group(fi_last, "iSPN", "LID on-state").fi_sum, "off-state (last sweep)", group(fi_last, "iSPN", "LID off-state").fi_sum, " spikes")


**Result: reproduced.** The 20 to 300 pA spike sum is a median 65 on-state against 118.5 off-state, P = 7.2e-05; on-state per-step medians are lower from 120 pA up. Keeping the last sweep of a repeated step gives the same (P = 7.2e-05).

## fig3d-ispn-sul-no-change

> Sulpiride does not alter the somatic excitability of on-state iSPNs.

Criterion: a claim of absence; two-sided Mann-Whitney test of rheobase, on-state against on-state with sulpiride; reproduced when P >= 0.05; a Wilcoxon test on cells recorded in both conditions is a variant.

In [ ]:
sul = group(fi, "iSPN", "LID on-state with sulpiride")
report("on-state", ion.rheobase, "on-state + sulpiride", sul.rheobase, " pA")
pair = ion.merge(sul, on=["path", "cell"], suffixes=("", "_sul"))
print("cells recorded in both:", len(pair), "| rheobase before", pair.rheobase.tolist(), "after", pair.rheobase_sul.tolist())
if len(pair) >= 2 and (pair.rheobase != pair.rheobase_sul).any():
    print("Wilcoxon (paired) P =", round(wilcoxon(pair.rheobase, pair.rheobase_sul)[1], 4))
# variant: a step recorded twice keeps its last sweep
report("on-state (last sweep)", group(fi_last, "iSPN", "LID on-state").rheobase, "on-state + sulpiride (last sweep)", group(fi_last, "iSPN", "LID on-state with sulpiride").rheobase, " pA")
report("on-state, spike sum", ion.fi_sum, "on-state + sulpiride, spike sum", sul.fi_sum)
report("on-state, spike sum (last sweep)", group(fi_last, "iSPN", "LID on-state").fi_sum, "on-state + sulpiride (last sweep)", group(fi_last, "iSPN", "LID on-state with sulpiride").fi_sum)


**Result: reproduced.** On-state rheobase median 160 pA (11 cells) and with sulpiride 120 pA (7 cells), P = 0.143 (paper P = 0.287). The effect is a 40 pA lower median with sulpiride, not significant; the 2 cells recorded in both conditions give Wilcoxon P = 0.5. Keeping the last sweep of a repeated step gives the same (P = 0.143), and the spike sums (median 65 on-state, 73 with sulpiride) give P = 0.60 with either sweep.

## fig6c-m1r-antag-somatic-unchanged

> M1R antagonism throughout the off-state does not have a strong impact on the elevated somatic excitability of off-state iSPNs.

Criterion: a claim of absence; under the paper's default, a two-sided Mann-Whitney test of rheobase, off-state control against M1R antagonist; reproduced when P >= 0.05. The sum of spike counts is a variant.

In [ ]:
veh, ant = group(fi, "iSPN", "LID off-state, vehicle control"), group(fi, "iSPN", "LID off-state with M1R antagonists")
report("vehicle", veh.rheobase, "M1R antagonists", ant.rheobase, " pA")
report("vehicle, spike sum", veh.fi_sum, "M1R antagonists, spike sum", ant.fi_sum)
by_animal(pd.concat([veh, ant]), "rheobase")
# variant: a step recorded twice keeps its last sweep
report("vehicle (last sweep)", group(fi_last, "iSPN", "LID off-state, vehicle control").rheobase, "M1R antagonists (last sweep)", group(fi_last, "iSPN", "LID off-state with M1R antagonists").rheobase, " pA")


**Result: reproduced.** Off-state rheobase median 120 pA with vehicle (8 cells) and 140 pA with M1R antagonists (10 cells), P = 0.68; the spike sums (139 and 111) give P = 0.45. Keeping the last sweep of a repeated step gives the same rheobase test (P = 0.68).

## fig7c-cdgi-ko-somatic-oscillation

> Deletion of CDGI does not blunt the oscillation of iSPN somatic excitability between on- and off-states.

Criterion: in CDGI KO iSPNs, a two-sided Mann-Whitney test of rheobase, off-state against on-state; reproduced when P < 0.05 with the on-state higher.

In [ ]:
koff, kon = group(fi, "iSPN", "CDGI knockout, LID off-state"), group(fi, "iSPN", "CDGI knockout, LID on-state")
report("KO on-state", kon.rheobase, "KO off-state", koff.rheobase, " pA")
report("KO on-state, spike sum", kon.fi_sum, "KO off-state, spike sum", koff.fi_sum)
by_animal(pd.concat([kon, koff]), "rheobase")
# variant: a step recorded twice keeps its last sweep
report("KO on-state (last sweep)", group(fi_last, "iSPN", "CDGI knockout, LID on-state").rheobase, "KO off-state (last sweep)", group(fi_last, "iSPN", "CDGI knockout, LID off-state").rheobase, " pA")


**Result: reproduced.** In CDGI KO iSPNs the on-state rheobase median is 210 pA (8 cells) against 140 pA off-state (13 cells), P = 0.0010; the spike sums give P = 0.0033. Keeping the last sweep of a repeated step gives the same (P = 0.0010).

## fig9d-m1r-crispr-somatic-unchanged

> Deletion of M1Rs from iSPNs does not alter the elevated somatic excitability of off-state iSPNs.

Criterion: a claim of absence; two-sided Mann-Whitney test of rheobase, interleaved control against M1R CRISPR; reproduced when P >= 0.05.

In [ ]:
ctl, cr = group(fi, "iSPN", "LID off-state, control (gRNA only)"), group(fi, "iSPN", "LID off-state, M1R deletion")
report("control (gRNA only)", ctl.rheobase, "M1R deletion", cr.rheobase, " pA")
report("control, spike sum", ctl.fi_sum, "M1R deletion, spike sum", cr.fi_sum)
by_animal(pd.concat([ctl, cr]), "rheobase")
# variant: a step recorded twice keeps its last sweep
report("control (last sweep)", group(fi_last, "iSPN", "LID off-state, control (gRNA only)").rheobase, "M1R deletion (last sweep)", group(fi_last, "iSPN", "LID off-state, M1R deletion").rheobase, " pA")
report("control, spike sum (last sweep)", group(fi_last, "iSPN", "LID off-state, control (gRNA only)").fi_sum, "M1R deletion (last sweep)", group(fi_last, "iSPN", "LID off-state, M1R deletion").fi_sum)


**Result: reproduced.** Rheobase median 160 pA in gRNA-only controls (6 cells) and 120 pA with M1R deletion (8 cells), P = 0.147; the spike sums give P = 0.24. The deletion group has the lower median, as an unchanged or raised excitability would. Keeping the last sweep of a repeated step gives the same (rheobase P = 0.147, spike sums P = 0.24).

## Dendritic excitability index: variants

The index follows the paper's script and the claims file (`lid.dendritic_index`). The variants below are shown with each dendritic claim: every proximal site paired with every distal site of the cell (pairing `all`), and no basal-ratio filter. No pair falls outside 0.5 to 2 (data check), so the filter drops nothing.

In [ ]:
den_all_pairs, _, _ = lid.dendritic_index(dendritic, pairing="all")
def den_test(ct, a, b, label_a, label_b):
    for name, df in [("claims file (paired by number)", den_cells), ("every proximal with every distal", den_all_pairs)]:
        print(name + ":")
        report(label_a, group(df, ct, a)["index"], label_b, group(df, ct, b)["index"])
print("ok")

## fig1i-dspn-dendritic-on-higher

> The dendritic excitability index of dSPNs is higher in the on-state than in the off-state.

Criterion: two-sided Mann-Whitney test of the per-cell index; reproduced when P < 0.05 and the on-state median is higher.

In [ ]:
den_test("dSPN", "LID on-state", "LID off-state", "on-state", "off-state")
by_animal(pd.concat([group(den_cells, "dSPN", "LID on-state"), group(den_cells, "dSPN", "LID off-state")]), "index")

**Result: reproduced.** Dendritic index median 0.476 on-state against 0.353 off-state (10 cells each), P = 0.0091 (paper P = 0.0115); pairing every proximal with every distal site gives P = 0.017.

## fig1i-dspn-dendritic-sch-lower

> SCH 23390 reverses the higher dendritic excitability of on-state dSPNs.

Criterion: two-sided Mann-Whitney test, on-state against on-state with SCH; reproduced when P < 0.05 and the SCH median is lower.

In [ ]:
den_test("dSPN", "LID on-state", "LID on-state with SCH 23390", "on-state", "on-state + SCH")
pair = group(den_cells, "dSPN", "LID on-state").merge(group(den_cells, "dSPN", "LID on-state with SCH 23390"), on=["path", "cell"], suffixes=("", "_sch"))
print("cells with an index in both conditions:", len(pair))
if len(pair) >= 2:
    print(pair[["date", "cell", "index", "index_sch"]].round(3).to_string())
    print("Wilcoxon (paired) P =", round(wilcoxon(pair["index"], pair.index_sch)[1], 4))
by_animal(pd.concat([group(den_cells, "dSPN", "LID on-state"), group(den_cells, "dSPN", "LID on-state with SCH 23390")]), "index")
# The dandiset description (item 3): "The first pair of dendritic recording sites of the dSPN recorded on 2019-04-11
# (cell 1, on-state with SCH23390, Figure 1I). Only its second pair is available." Per cell, the SCH group against the
# paper's Figure 1I table (rows in date and cell order, as in link-dendritic-cells):
t1 = tables["Fig 1"]["Fig 1I_dSPN dendritic excitabil"].set_axis(["id", "group", "v"], axis=1)
on_p, sch_p = t1[t1.group == "LID on"].v, t1[t1.group == "LID on + SCH"].v
D = group(den_cells, "dSPN", "LID on-state with SCH 23390").sort_values(["date", "cell"]).reset_index(drop=True)
D["paper"] = sch_p.to_numpy()[:len(D)]
D["incomplete_in_dandiset"] = (D.date == "20190411") & (D.cell == 1)
print(D[["date", "cell", "n_pairs", "index", "paper", "incomplete_in_dandiset"]].round(3).to_string())
print("\nThe paper's own table, on-state against SCH (exact Mann-Whitney):")
print(f"  P = {stats.mannwhitneyu(on_p, sch_p, method='exact').pvalue:.4f} (medians {on_p.median():.3f} and {sch_p.median():.3f})")
print("The dandiset without the incomplete cell:")
keep = ~((D.date == "20190411") & (D.cell == 1))
report("on-state", group(den_cells, "dSPN", "LID on-state")["index"], "on-state + SCH, without 2019-04-11 cell 1", D[keep]["index"])
sa = group(den_all_pairs, "dSPN", "LID on-state with SCH 23390")
report("on-state (every proximal with every distal)", group(den_all_pairs, "dSPN", "LID on-state")["index"],
       "on-state + SCH, without 2019-04-11 cell 1", sa[~((sa.date == "20190411") & (sa.cell == 1))]["index"])
print("The paper's table without its value for that cell:")
report("paper on-state", on_p, "paper SCH, without its first row", sch_p.iloc[1:])


**Result: reproduced, resting on an assumption (that the incomplete cell is left out).** With every cell the dandiset holds, the SCH 23390 median (0.419, 9 cells) is lower than the on-state median (0.476, 10 cells) but P = 0.066, and 0.079 with every proximal site paired with every distal one. One of the nine SCH cells, 2019-04-11 cell 1, is incomplete in the dandiset: its description says the first of its two site pairs is not deposited. From its second pair alone its index is 0.584, against the paper's 0.228 for the first row of its SCH table; the other eight cells are within 10% of the paper's rows. The paper's own table gives exact P = 0.0133, the paper's value. Without the incomplete cell the dandiset gives P = 0.027 (SCH median 0.384, 8 cells), and 0.034 with every proximal paired with every distal site, both with the SCH median lower. The outcome rests on leaving out the cell whose index the dandiset cannot give as the paper measured it; with it, the test fails. No cell has an index in both conditions, and the on-state cells come from 2017 mice and the SCH cells from 2019 mice.

## fig3f-ispn-dendritic-on-lower

> The dendritic excitability index of iSPNs falls in the on-state compared with the off-state.

Criterion: two-sided Mann-Whitney test of the per-cell index; reproduced when P < 0.05 and the on-state median is lower.

In [ ]:
den_test("iSPN", "LID on-state", "LID off-state", "on-state", "off-state")
by_animal(pd.concat([group(den_cells, "iSPN", "LID on-state"), group(den_cells, "iSPN", "LID off-state")]), "index")

**Result: reproduced.** iSPN index median 0.340 on-state (9 cells) against 0.524 off-state (10 cells), P = 0.0128 (paper P = 0.0029, with 10 on-state cells); the other pairing gives P = 0.016.

## fig3f-ispn-dendritic-sul-no-change

> Sulpiride does not significantly alter dendritic excitability in on-state iSPNs.

Criterion: a claim of absence, two-sided Mann-Whitney test, on-state against on-state with sulpiride; reproduced when P >= 0.05.

In [ ]:
den_test("iSPN", "LID on-state", "LID on-state with sulpiride", "on-state", "on-state + sulpiride")

**Result: reproduced.** On-state median 0.340 (9 cells) and with sulpiride 0.497 (8 cells), P = 0.093 (paper P = 0.0545); the other pairing gives P = 0.139.

## fig6d-m1r-antag-dendritic-lower

> M1R antagonism prevents the off-state elevation of iSPN dendritic excitability.

Criterion: two-sided Mann-Whitney test of the index, control against M1R antagonist; reproduced when P < 0.05 and the antagonist median is lower.

In [ ]:
den_test("iSPN", "LID off-state, vehicle control", "LID off-state with M1R antagonists", "vehicle", "M1R antagonists")
by_animal(pd.concat([group(den_cells, "iSPN", "LID off-state, vehicle control"), group(den_cells, "iSPN", "LID off-state with M1R antagonists")]), "index")

**Result: reproduced.** Index median 0.451 with vehicle (9 cells) against 0.269 with M1R antagonists (7 cells), P = 0.023 (paper P = 0.012); the other pairing gives P = 0.0079.

## fig7f-cdgi-ko-dendritic-unchanged

> The off-state elevation of iSPN dendritic excitability is absent in CDGI KO mice (off against on not different), while it is present in wild type.

Criterion: a claim of absence for the KO, two-sided Mann-Whitney test of the index, KO off-state against KO on-state; reproduced when P >= 0.05. The wild-type part is fig3f-ispn-dendritic-on-lower.

In [ ]:
den_test("iSPN", "CDGI knockout, LID off-state", "CDGI knockout, LID on-state", "KO off-state", "KO on-state")
by_animal(pd.concat([group(den_cells, "iSPN", "CDGI knockout, LID off-state"), group(den_cells, "iSPN", "CDGI knockout, LID on-state")]), "index")

**Result: reproduced.** In CDGI KO iSPNs the index median is 0.328 off-state (10 cells) and 0.286 on-state (8 cells), P = 0.32 (paper P = 0.395); the other pairing gives P = 0.63.

## Sr2+-oEPSCs: readings

The claims below have no criterion beyond the paper's test: a two-sided Mann-Whitney test of per-cell values (Methods). The detector is a stand-in for TaroTools (see `lid.detect_events`), so each claim is run under the readings that could change it: thresholds of 5 SD (the Methods), 4 and 3 SD, and the per-cell amplitude as the mean of the cell's events or as their median (the paper's table holds medians of its event lists, link-sr-cells).

In [ ]:
def sr_test(ct, col_fmt, higher=None):
    rows = []
    for n in (5, 4, 3):
        for stat in (["mean_amp", "median_amp"] if "amp" in col_fmt else ["freq"]):
            col = f"{stat}_{n}"
            a = sr[(sr.cell_type == ct) & (sr.condition == "LID on-state")][col]
            b = sr[(sr.cell_type == ct) & (sr.condition == "LID off-state")][col]
            u, p = mw(a, b)
            rows.append(dict(threshold_sd=n, measure=stat, n_on=len(a), n_off=len(b), median_on=a.median(), median_off=b.median(),
                             diff=a.median() - b.median(), U=u, P=p))
    out = pd.DataFrame(rows)
    print(out.round(4).to_string())
    return out
print("ok")

## fig2h-dspn-sr-amplitude-on-larger

> The amplitude of Sr2+-oEPSCs evoked from cortical afferents is larger in on-state than in off-state dSPNs.

Test (paper's default): two-sided Mann-Whitney test of per-cell amplitude, on-state against off-state; reproduced when P < 0.05 with the on-state larger.

In [ ]:
_ = sr_test("dSPN", "amp")
print(sr[sr.cell_type == "dSPN"][["id", "condition", "subject", "n_sweeps", "n_events_5", "mean_amp_5", "median_amp_5"]].round(2).to_string())
t_sr = tables["Fig 2"]["Fig 2H_dSPN Sr-oEPSC amp"].set_axis(["id", "group", "v"], axis=1)
a_, b_ = t_sr[t_sr.group == "LID on"].v, t_sr[t_sr.group == "LID off"].v
print("the paper's own per-cell values (Fig 2H_dSPN Sr-oEPSC amp):")
report("on-state", a_, "off-state", b_)
print(f"  exact Mann-Whitney P = {stats.mannwhitneyu(a_, b_, method='exact').pvalue:.4f}")


**Result: reproduced, resting on an assumption (the event detector).** At 5 SD the per-cell mean amplitude is a median 15.6 pA on-state (8 cells) against 10.1 pA off-state (10 cells), P = 0.0031 (paper P = 0.0003); with the per-cell median, as the paper's table holds, 12.5 against 8.4 pA, P = 0.0044. Five of the six readings give P < 0.05 with the on-state larger; the sixth (3 SD, median) gives P = 0.068. On the paper's own per-cell values (its table) the on-state is larger too, P = 0.0003.

## fig2i-dspn-sr-frequency-unchanged

> The frequency of Sr2+-oEPSCs in dSPNs does not differ between states.

Test: a claim of absence; two-sided Mann-Whitney test of per-cell frequency; reproduced when P >= 0.05.

In [ ]:
_ = sr_test("dSPN", "freq")
t_sr = tables["Fig 2"]["Fig 2I_dSPN Sr-oEPSC freq"].set_axis(["id", "group", "v"], axis=1)
a_, b_ = t_sr[t_sr.group == "LID on"].v, t_sr[t_sr.group == "LID off"].v
print("the paper's own per-cell values (Fig 2I_dSPN Sr-oEPSC freq):")
report("on-state", a_, "off-state", b_)
print(f"  exact Mann-Whitney P = {stats.mannwhitneyu(a_, b_, method='exact').pvalue:.4f}")


**Result: reproduced, resting on an assumption (the event detector).** Frequency median 28.6 Hz on-state against 23.1 Hz off-state at 5 SD, P = 0.122 (paper P = 0.127); at 4 and 3 SD P = 0.0505 and 0.068. No reading reaches P < 0.05, but the 4 SD reading is at the edge, and the detector finds a median 1.33 times the paper's events (link-sr-cells), so the margin is thin. On the paper's own per-cell frequencies the test gives P = 0.130 (exact 0.146), on-state 20.1 Hz against 16.5 Hz, so the absence holds on the paper's values as on the detector's.

## fig4g-ispn-sr-amplitude-on-smaller

> The amplitude of Sr2+-oEPSCs is smaller in on-state than in off-state iSPNs.

Test: two-sided Mann-Whitney test of per-cell amplitude; reproduced when P < 0.05 with the on-state smaller.

In [ ]:
_ = sr_test("iSPN", "amp")
print(sr[sr.cell_type == "iSPN"][["id", "condition", "subject", "n_sweeps", "n_events_5", "mean_amp_5", "median_amp_5"]].round(2).to_string())
t_sr = tables["Fig 4"]["Fig 4G_iSPN Sr-oEPSC amp"].set_axis(["id", "group", "v"], axis=1)
a_, b_ = t_sr[t_sr.group == "LID on"].v, t_sr[t_sr.group == "LID off"].v
print("the paper's own per-cell values (Fig 4G_iSPN Sr-oEPSC amp):")
report("on-state", a_, "off-state", b_)
print(f"  exact Mann-Whitney P = {stats.mannwhitneyu(a_, b_, method='exact').pvalue:.4f}")


**Result: reproduced, resting on an assumption (the event detector).** At 5 SD the per-cell mean amplitude is a median 8.2 pA on-state (8 cells) against 11.2 pA off-state (7 cells), P = 0.0093 (paper P = 0.0003); every reading gives P < 0.05 with the on-state smaller (P = 0.0059 to 0.040). On the paper's own per-cell values the on-state is smaller too, P = 0.0003.

## fig4h-ispn-sr-frequency-unchanged

> The frequency of Sr2+-oEPSCs in iSPNs does not differ between states.

Test: a claim of absence; two-sided Mann-Whitney test of per-cell frequency; reproduced when P >= 0.05.

In [ ]:
_ = sr_test("iSPN", "freq")
t_sr = tables["Fig 4"]["Fig 4H_iSPN Sr-oEPSC freq"].set_axis(["id", "group", "v"], axis=1)
a_, b_ = t_sr[t_sr.group == "LID on"].v, t_sr[t_sr.group == "LID off"].v
print("the paper's own per-cell values (Fig 4H_iSPN Sr-oEPSC freq):")
report("on-state", a_, "off-state", b_)
print(f"  exact Mann-Whitney P = {stats.mannwhitneyu(a_, b_, method='exact').pvalue:.4f}")


**Result: reproduced, resting on an assumption (the event detector).** Frequency median 22.2 Hz on-state against 32.2 Hz off-state at 5 SD, P = 0.12 (paper P = 0.23); 4 and 3 SD give P = 0.12 and 0.19. On the paper's own per-cell frequencies P = 0.23 (on-state 19.0 Hz, off-state 20.3 Hz), as the detector's readings.

## GRAB-ACh3.0: variants

Normalised single-pulse AUC per region (`lid.grab_auc`). Shown with each claim: Fmin as the Methods give it (TTX or the quinpirole baseline, whichever is smaller), TTX only (the script), and the whole-frame brightness Prairie View computed during recording (`roi_response_grabach3`) in place of the 50 um circle.

In [ ]:
grab_v = {"circle, Fmin = min(TTX, quinpirole)": grab,
          "circle, Fmin = TTX": lid.grab_auc(grab_trials, fmin_rule="ttx"),
          "whole frame, Fmin = min(TTX, quinpirole)": lid.grab_auc(grab_trials, signal="whole")}
def gvals(df, state, drug):
    return df[(df.state == state) & (df.drug == drug)]
def g_unpaired(sa, sb, drug="control"):
    for name, df in grab_v.items():
        print(name + ":")
        report(sa, gvals(df, sa, drug).auc, sb, gvals(df, sb, drug).auc)
def g_paired(state, drug):
    for name, df in grab_v.items():
        a = gvals(df, state, "control").set_index("region" if False else ["path", "region"]).auc
        b = gvals(df, state, drug).set_index(["path", "region"]).auc
        j = pd.concat([a, b], axis=1, keys=["nodrug", "drug"]).dropna()
        w, p = wilcoxon(j.nodrug, j.drug)
        print(f"{name}: {state}, {drug}: regions {len(j)}, median no drug {j.nodrug.median():.3f}, with drug {j.drug.median():.3f}, "
              f"median paired difference {np.median(j.drug - j.nodrug):.3f}, drug higher in {int((j.drug > j.nodrug).sum())}; Wilcoxon W={w:.1f}, P={p:.4g}")
print(grab[grab.drug == "control"].groupby(["state", "subject"]).auc.agg(["count", "median"]).round(3).to_string())

## fig5f-ach-off-elevated

> Evoked ACh release (GRAB-ACh3.0) in slices from LID off-state mice is greater than in unlesioned striatum.

Criterion: single-pulse normalised area without drug, one value per region (ROI), as the paper's n counts ROIs; two-sided Mann-Whitney test, unlesioned against off-state; reproduced when P < 0.05 with the off-state higher. A reader amended the criterion after this attempt asked about its unit (questions/zhai-2025.yaml): it said "one region per mouse", and now tests regions, the units of the legend of Figure 5F ("n = 13 regions of interest (ROIs) from three mice ... n = 11 ROIs from five mice"). One value per mouse, the mean of its regions, is shown after the per-region test as a variant.

In [ ]:
g_unpaired("UnlesionedControl", "OffState")
print()
for name, df in grab_v.items():
    pm = gvals(df, "UnlesionedControl", "control").groupby("subject").auc.mean(), gvals(df, "OffState", "control").groupby("subject").auc.mean()
    print(name + ", variant, one value per mouse:")
    report("unlesioned", pm[0], "off-state", pm[1])

**Result: reproduced.** Per region, the criterion's unit, the off-state median is 0.406 (11 regions) against 0.111 unlesioned (13 regions), U = 1, P = 5.0e-05 (paper P < 0.0001); with the TTX-only Fmin P = 8.2e-05 (10 off-state regions) and with the whole frame P = 5.0e-05. The variant with one value per mouse gives 0.388 (5 mice) against 0.131 (3 mice), P = 0.036, the smallest P a 3 against 5 test can give, under all three signals.

## fig5f-ach-off-like-6ohda

> Evoked ACh release in the off-state is similar to that after a 6-OHDA lesion.

Criterion: a claim of absence; two-sided Mann-Whitney test of the single-pulse normalised area without drug, one value per region (ROI), 6-OHDA against off-state; reproduced when P >= 0.05. The reader's answer to this attempt's question on fig5f-ach-off-elevated added "one value per region (ROI)" to this criterion; the test already used regions.

In [ ]:
g_unpaired("LesionedControl", "OffState")

**Result: reproduced.** 6-OHDA median 0.315 (10 regions) and off-state 0.406 (11 regions), P = 0.22 (paper P = 0.4679); with the TTX-only Fmin P = 0.79, with the whole frame P = 0.22.

## fig5f-ach-6ohda-elevated

> Evoked ACh release is markedly elevated in 6-OHDA-lesioned striatum compared with unlesioned striatum.

Test (paper's default): two-sided Mann-Whitney test of regions, unlesioned against 6-OHDA; reproduced when P < 0.05 with 6-OHDA higher.

In [ ]:
g_unpaired("UnlesionedControl", "LesionedControl")

**Result: reproduced.** 6-OHDA median 0.315 (10 regions) against 0.111 unlesioned (13 regions), P = 0.00022 (paper P < 0.0001); all three variants give P < 0.0003.

## fig5f-ach-d2-control

> In unlesioned striatum sulpiride increases and quinpirole suppresses evoked ACh release.

Test (paper's default for paired data): two-sided Wilcoxon tests of each region with the drug against the same region without drug; reproduced when sulpiride is higher and quinpirole lower, each with P < 0.05.

In [ ]:
g_paired("UnlesionedControl", "10 µM sulpiride")
g_paired("UnlesionedControl", "10 µM quinpirole")

**Result: reproduced.** In the 13 unlesioned regions sulpiride raises the area in every region (median 0.111 to 0.212, Wilcoxon P = 0.00024; paper P = 0.0005) and quinpirole lowers it in every region (0.111 to 0.036, P = 0.00024; paper P = 0.0034). The variants give the same.

## fig5f-ach-da-suppresses

> Dopamine (50 nM) strongly suppresses evoked ACh release in 6-OHDA-lesioned and off-state striatum, and sulpiride has no effect after the lesion.

Test (paper's default): two-sided Wilcoxon tests, dopamine against no drug in 6-OHDA and in off-state regions (reproduced when lower with P < 0.05), and sulpiride against no drug in 6-OHDA regions (a claim of absence, P >= 0.05).

In [ ]:
g_paired("LesionedControl", "50 nM dopamine")
g_paired("OffState", "50 nM dopamine")
g_paired("LesionedControl", "10 µM sulpiride")

**Result: reproduced.** Dopamine lowers the area in all 10 6-OHDA regions (median 0.315 to 0.103, P = 0.0020; paper P = 0.002) and all 11 off-state regions (0.406 to 0.082, P = 0.00098; paper P = 0.001). Sulpiride in 6-OHDA regions: median paired difference -0.067, higher in 4 of 10, P = 0.92 (paper P = 0.5566). The variants give the same.

## fig10b-m1r-aims

> AIMs are attenuated by deletion of M1Rs in iSPNs.

Test (Methods): repeated-measures two-way ANOVA (group between mice, session within), on per-session scores summed over the session's time points (the claims file's reading of the total, link-aim-mice). The claim is judged on the group effect, P < 0.05 with M1R CRISPR lower, for the total and for each of axial, limb and orolingual scores, as the paper reports. The group F test does not depend on a sphericity correction.

In [ ]:
def mixed_anova(df, value):
    # df: one row per mouse and session, columns mouse, group, session, value
    w = df.pivot_table(index=["group", "mouse"], columns="session", values=value)
    w = w.dropna()
    grand = w.values.mean()
    subj = w.mean(axis=1)
    n_s = w.shape[1]
    g_means = subj.groupby(level=0).mean()
    n_g = subj.groupby(level=0).size()
    ss_g = n_s * sum(n_g[g] * (g_means[g] - grand) ** 2 for g in g_means.index)
    ss_subj = n_s * sum(((subj.loc[g] - g_means[g]) ** 2).sum() for g in g_means.index)
    df_g, df_s = len(g_means) - 1, len(subj) - len(g_means)
    F = (ss_g / df_g) / (ss_subj / df_s)
    return F, df_g, df_s, stats.f.sf(F, df_g, df_s), w.groupby(level=0).mean().mean(axis=1), len(subj), w.shape[1]

f10 = aim[(aim.figure == 10) & aim.behavior.isin(["axial", "limb", "orolingual"])]
per = f10.groupby(["group", "mouse", "session", "behavior"]).score.sum().unstack("behavior").reset_index()
per["total"] = per[["axial", "limb", "orolingual"]].sum(axis=1)
for v in ["total", "axial", "limb", "orolingual"]:
    F, d1, d2, p, gm, n, ns = mixed_anova(per, v)
    print(f"{v}: group F({d1},{d2}) = {F:.2f}, P = {p:.3g}; mean per session {gm.round(2).to_dict()} ({n} mice, {ns} sessions)")
print(per.groupby(["group", "session"]).total.mean().unstack().round(1).to_string())
lab = aim[(aim.figure == 10) & (aim.behavior == "total")].groupby(["group", "mouse", "session"]).score.sum().rename("lab_total").reset_index()
F, d1, d2, p, gm, n, ns = mixed_anova(lab, "lab_total")
print(f"the lab's own total column: group F({d1},{d2}) = {F:.2f}, P = {p:.3g}; mean per session {gm.round(2).to_dict()}")
pt = tables["Fig 10"]["Fig 10B_total AIMs"].set_axis(["mouse", "group", "session", "total"], axis=1)
F, d1, d2, p, gm, n, ns = mixed_anova(pt, "total")
print(f"the paper's table: group F({d1},{d2}) = {F:.2f}, P = {p:.3g} ({n} mice, {ns} sessions)")


**Result: reproduced.** On the dandiset's Figure 10 sessions (11 control, 10 M1R CRISPR mice, 5 sessions) the group effect is F(1,19) = 28.70, P = 3.6e-05 for the total (paper 29.30), 13.80 for axial (P = 0.0015), 41.43 for limb and 44.58 for orolingual (P below 4e-06), each with M1R CRISPR lower (total 26.1 against 41.9 per session); the axial, limb and orolingual F values equal the paper's. The link holds for every mouse of this figure. The total here is the sum of the three scores, as the claims file states; with the lab's own total column, which differs from that sum at a few time points, the group F is 29.30 (P = 3.2e-05), equal to the paper's, as is the F from the paper's own table.

## Claims tested on the paper's tables (data: elsewhere)

The claims below rest on data the dandiset does not hold (control and 6-OHDA patch-clamp recordings, rotation counts, open field), which the claims file places in the paper's tables at Zenodo doi:10.5281/zenodo.17307059. They are tested there and counted apart.

## fig1-dspn-lesion-and-off-somatic

> dSPN somatic excitability (rheobase) rises after the 6-OHDA lesion and falls in the LID off-state.

Criterion (claims file): On the paper's per-cell rheobase table, two-sided Mann-Whitney tests of control against 6-OHDA (lower rheobase after the lesion) and 6-OHDA against off-state (higher in the off-state). Reproduced when both have P < 0.05 in the stated direction.

In [ ]:
def tab(book, sheet):
    return tables[book][sheet]
r1 = tab("Fig 1", "Fig 1F_dSPN rheobase").set_axis(["id", "group", "v"], axis=1)
print("fig1-dspn-lesion-and-off-somatic (rheobase, pA):")
report("6-OHDA", r1[r1.group == "6-OHDA"].v, "control", r1[r1.group == "control"].v, " pA")
report("off-state", r1[r1.group == "LID off"].v, "6-OHDA", r1[r1.group == "6-OHDA"].v, " pA")

**Result: reproduced (paper's table).** dSPN rheobase median 120 pA after 6-OHDA against 220 pA in controls, P = 0.00030, and 200 pA off-state against 120 pA after 6-OHDA, P = 9.5e-05, both in the stated direction (paper P < 0.0001 for both).

## fig1-dspn-lesion-dendritic-unchanged

> The dSPN dendritic excitability index is not changed by the 6-OHDA lesion or in the off-state.

Criterion (claims file): A claim of absence, on the paper's per-cell table, two-sided Mann-Whitney tests of control against 6-OHDA and 6-OHDA against off-state. Reproduced when both have P >= 0.05.

In [ ]:
d1 = tab("Fig 1", "Fig 1I_dSPN dendritic excitabil").set_axis(["id", "group", "v"], axis=1)
print("fig1-dspn-lesion-dendritic-unchanged (index):")
report("control", d1[d1.group == "control"].v, "6-OHDA", d1[d1.group == "6-OHDA"].v)
report("6-OHDA", d1[d1.group == "6-OHDA"].v, "off-state", d1[d1.group == "LID off"].v)

**Result: reproduced (paper's table).** Control against 6-OHDA P = 0.82 (medians 0.358 and 0.392) and 6-OHDA against off-state P = 0.90 (0.392 and 0.333), as the paper (0.824 and 0.905).

## fig3-ispn-lesion-and-off-somatic

> iSPN somatic excitability is lowered by the 6-OHDA lesion and restored in the LID off-state.

Criterion (claims file): On the paper's table, two-sided Mann-Whitney tests of control against 6-OHDA (higher rheobase after the lesion) and 6-OHDA against off-state (lower in the off-state). Reproduced when both have P < 0.05 in the stated direction.

In [ ]:
r3 = tab("Fig 3", "Fig 3D_iSPN rheobase").set_axis(["id", "group", "v"], axis=1)
print("fig3-ispn-lesion-and-off-somatic (rheobase, pA):")
report("6-OHDA", r3[r3.group == "6-OHDA"].v, "control", r3[r3.group == "Control"].v, " pA")
report("off-state", r3[r3.group == "LID off"].v, "6-OHDA", r3[r3.group == "6-OHDA"].v, " pA")

**Result: reproduced (paper's table).** iSPN rheobase median 200 pA after 6-OHDA against 120 pA in controls, P = 0.00040, and 130 pA off-state against 200 pA after 6-OHDA, P = 0.00061 (paper P = 0.002 for both).

## fig3-ispn-lesion-dendritic-up

> The iSPN dendritic excitability index rises after the lesion and stays raised in the off-state.

Criterion (claims file): On the paper's table, two-sided Mann-Whitney tests of control against 6-OHDA and control against off-state. Reproduced when both have P < 0.05 with the later group higher.

In [ ]:
d3 = tab("Fig 3", "Fig 3F_iSPN dendritic excitabil").set_axis(["id", "group", "v"], axis=1)
print("fig3-ispn-lesion-dendritic-up (index):")
report("6-OHDA", d3[d3.group == "6-OHDA"].v, "control", d3[d3.group == "Control"].v)
report("off-state", d3[d3.group == "LID off"].v, "control", d3[d3.group == "Control"].v)

**Result: reproduced (paper's table).** iSPN index median 0.516 after 6-OHDA against 0.405 in controls, P = 0.035, and 0.519 off-state against 0.405, P = 0.024 (paper P = 0.0327 and 0.0220).

## fig7e-oxom-cdgi

> Oxotremorine-M increases bAP invasion of iSPN dendrites in wild-type but not in CDGI KO mice.

Criterion (claims file): On the paper's table, two-sided Wilcoxon tests before against after oxo-M. Reproduced when wild type has P < 0.05 with oxo-M higher and KO has P >= 0.05.

In [ ]:
e7 = tab("Fig 7", "Fig 7E_oxoM effect on DE").set_axis(["id", "genotype", "treatment", "v"], axis=1)
print("fig7e-oxom-cdgi (index before and after oxo-M, paired by experiment number):")
for gt in ["WT", "KO"]:
    w = e7[e7.genotype == gt].pivot_table(index="id", columns="treatment", values="v")
    s, p = wilcoxon(w["before (control)"], w["oxoM"])
    print(f"{gt}: n={len(w)}, median before {w['before (control)'].median():.3f}, after {w['oxoM'].median():.3f}, "
          f"median paired difference {np.median(w['oxoM'] - w['before (control)']):.3f}; Wilcoxon W={s:.1f}, P={p:.4g}")

**Result: reproduced (paper's table).** In wild type oxo-M raises the index (median 0.214 to 0.335, Wilcoxon P = 0.0098, as the paper); in CDGI KO it does not (0.295 to 0.324, median paired difference -0.045, P = 0.47; paper P = 0.910). The table pairs before and after values by experiment number; the P for the KO differs from the paper's, which may pair them differently.

## fig8a-cdgi-rotations

> Levodopa induces more contralateral rotations in CDGI KO mice than in wild type at the fifth dose.

Criterion (claims file): On the paper's per-mouse table, a two-sided Mann-Whitney test, KO against wild type. Reproduced when P < 0.05 with KO higher.

In [ ]:
a8 = tab("Fig 8", "Fig 8A_rotations 5th").set_axis(["id", "group", "v"], axis=1)
print("fig8a-cdgi-rotations (contralateral rotations in 30 s, fifth session):")
report("CDGI KO", a8[a8.group == "KO"].v, "wild type", a8[a8.group == "WT"].v)

**Result: reproduced (paper's table).** CDGI KO median 8.5 rotations in 30 s (10 mice) against 4.25 in wild type (12 mice), P = 0.012 (paper P = 0.0092 in the text).

## fig10a-m1r-rotations

> Levodopa induces more contralateral rotations in mice lacking M1Rs in iSPNs.

Criterion (claims file): On the paper's per-mouse table, a two-sided Mann-Whitney test, M1R deletion against control. Reproduced when P < 0.05 with the deletion higher.

In [ ]:
a10 = tab("Fig 10", "Fig 10A_rotations in M1R CRISPR").set_axis(["id", "group", "v"], axis=1)
print("fig10a-m1r-rotations (contralateral rotations in 30 s, fifth session):")
report("M1R deletion", a10[a10.group == '"-M1R"'].v, "control", a10[a10.group == "control"].v)

**Result: reproduced (paper's table).** M1R deletion median 8.5 rotations (10 mice) against 6 in controls (11 mice), P = 0.0078 (paper P = 0.0056).

## fig10ef-open-field

> In MCI-Park mice, a low dose of levodopa increases locomotion profoundly only when M1Rs are deleted from the striatum; M1R deletion alone does not.

Criterion (claims file): On the paper's per-mouse tables, two-sided Mann-Whitney tests for distance and time mobile. Reproduced when control against MCI-Park and levodopa against levodopa with M1R deletion have P < 0.05 in the stated direction, and MCI-Park against M1R deletion alone has P >= 0.05, for both measures.

In [ ]:
print("fig10ef-open-field:")
for sheet in ["Fig 10E_distance traveled", "Fig 10F_time mobile"]:
    t = tab("Fig 10", sheet).set_axis(["id", "group", "v"], axis=1)
    g = lambda k: t[t.group == k].v
    print(sheet)
    report("MCI-Park", g("MCI-Park"), "control", g("control"))
    report("MCI-Park -M1R", g('"-M1R"'), "MCI-Park", g("MCI-Park"))
    report("+levodopa, -M1R", g('"+L-dopa, -M1R"'), "+levodopa", g('"+L-dopa"'))

**Result: reproduced (paper's table).** Distance: MCI-Park 4.68 m against 10.35 m in controls (P = 1.1e-05), MCI-Park with M1R deletion 3.97 m against MCI-Park (P = 0.63; paper 0.634), levodopa with deletion 28.1 m against levodopa alone 6.5 m (P = 0.00033). Time mobile: 112 against 190 s (P = 7.0e-06), 101 against 112 s (P = 0.47; paper 0.4657), 276 against 128 s (P = 0.00057). All six tests fall as the criterion requires.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| link-somatic-cells | partially reproduced |
| link-dendritic-cells | reproduced |
| link-sr-cells | partially reproduced |
| link-grab-rois | not reproduced |
| link-aim-mice | not reproduced |
| fig1f-dspn-rheobase-on-lower | reproduced |
| fig1e-dspn-fi-on-higher | reproduced |
| fig1f-dspn-sch-no-change | reproduced |
| fig1i-dspn-dendritic-on-higher | reproduced |
| fig1i-dspn-dendritic-sch-lower | reproduced (assumption: the incomplete cell 2019-04-11 cell 1 is left out) |
| fig2h-dspn-sr-amplitude-on-larger | reproduced (assumption: detector) |
| fig2i-dspn-sr-frequency-unchanged | reproduced (assumption: detector) |
| fig3d-ispn-rheobase-on-higher | reproduced |
| fig3c-ispn-fi-on-lower | reproduced |
| fig3d-ispn-sul-no-change | reproduced |
| fig3f-ispn-dendritic-on-lower | reproduced |
| fig3f-ispn-dendritic-sul-no-change | reproduced |
| fig4g-ispn-sr-amplitude-on-smaller | reproduced (assumption: detector) |
| fig4h-ispn-sr-frequency-unchanged | reproduced (assumption: detector) |
| fig5f-ach-off-elevated | reproduced |
| fig5f-ach-off-like-6ohda | reproduced |
| fig5f-ach-6ohda-elevated | reproduced |
| fig5f-ach-d2-control | reproduced |
| fig5f-ach-da-suppresses | reproduced |
| fig6c-m1r-antag-somatic-unchanged | reproduced |
| fig6d-m1r-antag-dendritic-lower | reproduced |
| fig7c-cdgi-ko-somatic-oscillation | reproduced |
| fig7f-cdgi-ko-dendritic-unchanged | reproduced |
| fig9d-m1r-crispr-somatic-unchanged | reproduced |
| fig10b-m1r-aims | reproduced |
| fig1-dspn-lesion-and-off-somatic (paper's table) | reproduced |
| fig1-dspn-lesion-dendritic-unchanged (paper's table) | reproduced |
| fig3-ispn-lesion-and-off-somatic (paper's table) | reproduced |
| fig3-ispn-lesion-dendritic-up (paper's table) | reproduced |
| fig7e-oxom-cdgi (paper's table) | reproduced |
| fig8a-cdgi-rotations (paper's table) | reproduced |
| fig10a-m1r-rotations (paper's table) | reproduced |
| fig10ef-open-field (paper's table) | reproduced |
| fig2bc, fig4bc, fig6f, fig7h, fig9f (2PLSM spine density) | blocked: methods underspecified |
| fig4k-confocal-spines | blocked: methods underspecified |
| fig8bc-cdgi-aims | blocked: data missing (the published Figure 8 scores and five of its mice are not in the dandiset) |

How the analysis differs from the paper's:
- Groups come from the files' `condition` column; the claims file names `condition_subfolder`, which this version does not have.
- Sr2+-oEPSC events come from a stand-in detector (`lid.detect_events`), not TaroTools with visual verification; it finds more events than the paper (median 1.33 times at 5 SD). The Sr2+ outcomes rest on it and are shown at 3, 4 and 5 SD and with the per-cell mean or median.
- GRAB-ACh3.0 areas use the Methods' 50 um circle on the stored movies; the paper used Prairie View's region. One region takes Fmax from the other region of its slice.
- For the SCH 23390 dendritic claim, 2019-04-11 cell 1 is left out: the dandiset holds only the second of its two site pairs, so its index (0.584) is not the paper's (0.228). With it, P = 0.066.
- A step recorded twice keeps its first sweep; keeping the last gives the same outcome for every somatic claim.
- The AIM ANOVA uses the group F of a mixed design; the paper's session effects with Geisser-Greenhouse correction are not tested.

What was found about the dataset:
- The draft holds the somatic and dendritic recordings of every on-state, off-state and drug group of Figures 1, 3, 6, 7 and 9 with the paper's cell counts, the Sr2+-oEPSC cells of Figures 2 and 4, all GRAB-ACh3.0 regions of Figure 5, the AIM sheets and videos of Figures 8 and 10, and 2PLSM spine Z-stacks. Control and 6-OHDA patch-clamp recordings, rotation counts and open-field data are not in it, as its description says. It holds 76 two-photon spine Z-stack files and the 16 confocal Z-stack files and overview image of Figure 4I to 4K, but no spine counts, and the description says the rows of the paper's spine tables cannot be matched to the files.
- The first site pair of dSPN 2019-04-11 cell 1 (on-state with SCH 23390) is not deposited, as the description says; its index from the second pair differs from the paper's, and with it the SCH 23390 dendritic test gives P = 0.066 against 0.027 without it.
- Two cells with dendritic line scans (2016-05-26 cell 3, CDGI KO off-state; 2016-06-08 cell 1, on-state) were imaged at one site only, at two spots, so they have no index, while the paper counts them.
- 27% of somatic cells do not match a row of the paper's table exactly, up to 24 spikes over 20 to 300 pA.
- The Figure 8 AIM sheets differ from the paper's table in most sessions, three of its mice are not in the paper and five of the paper's are missing, as the description says. Mouse 7830 has identical sheets for sessions 4 and 5, in the dandiset and in the paper's table.
- Mouse counts differ from the legends in four groups, which one subject per recording day can explain.
- The paper's per-cell Sr2+ "average" amplitude is the median of its event list.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")